# KKBox — Data Audit

## 1. Source files and audit scope

This notebook audits the complete KKBox churn dataset (v1 and v2 releases) before cleaning, aggregation, or modeling. Source files are never modified. Checks run in DuckDB; shared checks live in `src/audit_checks.py` and results are displayed with `src/audit_display.py`. The official description is summarized in [docs/source_dataset_notes.md](../docs/source_dataset_notes.md).

| Table (name in this notebook) | Source file | Role | Grain / key to assess |
|---|---|---|---|
| `train_v1` | `train.csv` | Labels, February 2017 expirations (documented) | One row per `msno` |
| `train_v2` | `train_v2.csv` | Labels, March 2017 expirations (documented) | One row per `msno` |
| `sub_zero` | `sample_submission_zero.csv` | Original test users, March 2017 (documented) | One row per `msno` |
| `sub_v2` | `sample_submission_v2.csv` | Test users, April 2017; example predictions are placeholders, not labels | One row per `msno` |
| `members` | `members_v3.csv` | Member profile attributes | One row per `msno` |
| `tx_v1`, `tx_v2` | `transactions.csv`, `transactions_v2.csv` | Subscription and payment records | Multiple records per member; no unique transaction key established |
| `logs_v1`, `logs_v2` | `user_logs.csv`, `user_logs_v2.csv` (converted to Parquet) | Daily listening summaries | One row per `msno` and `date` (verified for both releases) |

**Scope and evidence labels:** Full-file checks cover every row. `head()` and top-record displays are previews, not representative samples. The long-expiry investigation is a single-member case study. Documented claims (for example the label windows) are not treated as verified until checked against the local data.

**Revision status (2026-10-05):** Sections 2.7, 4.9 and 5.1–5.4 already cover v1 and v2 in DuckDB and will be moved onto the shared checks and display helper. Sections 2.1–2.6, 3, 4.1–4.8, 5.5 and 5.6 still contain earlier v2-only checks and are being replaced step by step.

**Reading order:** Setup → training and test-user files → members → transactions → user logs → conclusions.

**Run context:** Open this notebook from the repository's `notebooks/` directory. User logs are read from Parquet files on an external drive; update `PARQUET_DIR` in the setup cell on another machine. Large scans can take minutes.


### 1.1 Setup and file inventory

The setup registers every source as a DuckDB table or view without loading it into memory. Merged views (`train_all`, `tx_all`, `logs_all`) stack the two releases and add a `release` column; they never remove or de-duplicate rows. The inventory table confirms what was registered.

In [32]:
# Import necessary libraries
import sys
from pathlib import Path

import duckdb
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src import audit_checks as ac
from src.audit_display import show

In [33]:
# Paths, DuckDB connection, registered tables, and merged views
RAW_DIR = Path.cwd().parent / "data" / "raw"
try:
    # Machine-specific paths live in a git-ignored file: copy local_settings.example.py to local_settings.py
    from local_settings import PARQUET_DIR, TEMP_DIR
except ImportError:
    PARQUET_DIR = Path.cwd().parent / "data" / "processed" / "parquet"
    TEMP_DIR = Path.cwd().parent / "data" / "processed" / "duckdb_tmp"
Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute(f"SET temp_directory='{TEMP_DIR}'")


csv_tables = {
    "train_v1": "train.csv",
    "train_v2": "train_v2.csv",
    "sub_v2": "sample_submission_v2.csv",
    "sub_zero": "sample_submission_zero.csv",
    "members": "members_v3.csv",
    "tx_v1": "transactions.csv",
    "tx_v2": "transactions_v2.csv",
}
for name, filename in csv_tables.items():
    ac.register_csv(con, name, RAW_DIR / filename)

ac.register_parquet(con, "logs_v1", PARQUET_DIR / "user_logs.parquet")
ac.register_parquet(con, "logs_v2", PARQUET_DIR / "user_logs_v2.parquet")

for view, left, right in [("train_all", "train_v1", "train_v2"),
                          ("tx_all", "tx_v1", "tx_v2"),
                          ("logs_all", "logs_v1", "logs_v2")]:
    con.execute(f"""
        CREATE OR REPLACE VIEW {view} AS
        SELECT 'v1' AS release, * FROM {left}
        UNION ALL
        SELECT 'v2' AS release, * FROM {right}
    """)

In [34]:
# File inventory
sources = {name: RAW_DIR / filename for name, filename in csv_tables.items()}
sources["logs_v1"] = PARQUET_DIR / "user_logs.parquet"
sources["logs_v2"] = PARQUET_DIR / "user_logs_v2.parquet"

rows = []
for name, path in sources.items():
    summary = ac.table_summary(con, name)["value"]
    rows.append({"table": name, "file": path.name,
                 "size_mb": path.stat().st_size / 1e6,
                 "rows": summary["rows"], "columns": summary["columns"]})

inventory = pd.DataFrame(rows).set_index("table")
show("File inventory: every source registered for the audit", inventory,
     note="Sizes are on-disk sizes (user logs are the Parquet copies). Row counts are full-file counts.")

**File inventory: every source registered for the audit**  
*Sizes are on-disk sizes (user logs are the Parquet copies). Row counts are full-file counts.*

,file,size_mb,rows,columns
table,,,,
train_v1,train.csv,46.67,"992,931",2
train_v2,train_v2.csv,45.64,"970,960",2
sub_v2,sample_submission_v2.csv,42.65,"907,471",2
sub_zero,sample_submission_zero.csv,45.64,"970,960",2
members,members_v3.csv,427.92,"6,769,473",6
tx_v1,transactions.csv,"1,729.30","21,547,746",9
tx_v2,transactions_v2.csv,115.39,"1,431,009",9
logs_v1,user_logs.parquet,"6,776.85","392,106,543",9
logs_v2,user_logs_v2.parquet,774.26,"18,396,362",9


### 1.2 Supplied data documentation and churn-label logic

Field definitions were cross-checked against the official dataset description (summarized in [docs/source_dataset_notes.md](../docs/source_dataset_notes.md)) and `WSDMChurnLabeller.scala`. Observed schemas, ranges, and quality findings in this audit come from the local files, not from the documentation.

**Documented label windows (not yet verified locally):** `train.csv` covers users whose membership expires in February 2017 and `train_v2.csv` in March 2017. `sample_submission_zero.csv` is the original March test set; `sample_submission_v2.csv` is the April test set, whose true labels are not public.

The churn target is transaction-history based rather than cancellation based. A cancellation record may change the effective membership expiration date, but `is_cancel = 1` does not by itself imply churn.

The supplied label generator treats a user as retained when the next valid non-cancellation subscription begins less than 30 days after the effective expiration date. A gap of 30 days or more is treated as churn.

Detailed transaction ordering, eligibility-window logic, and label reconstruction rules are documented separately in [docs/churn_definition.md](../docs/churn_definition.md) and will be implemented and validated in the churn-definition phase.

**Documentation discrepancy:** The supplied member documentation references an `expiration_date` field, but the audited `members_v3.csv` contains only six columns and does not include it. No undocumented field is assumed.

### 1.3 Release map: how the v1 and v2 files relate

**Purpose:** one place that shows what each release contains, how the two relate, and how they are combined. Every figure is a full-data result from the section named in the last column. *Documented* means taken from the official description ([docs/source_dataset_notes.md](../docs/source_dataset_notes.md)) and not verified locally.

| Data | v1 | v2 | Relationship | Combination rule | Evidence |
|---|---|---|---|---|---|
| Label tables | `train.csv`: 992,931 users; 6.39% churn labels; documented February 2017 expirations | `train_v2.csv`: 970,960 users; 8.99% churn labels; documented March 2017 expirations | 881,701 users in both (88.80% of v1, 90.81% of v2); the label differs for 45,990 of them | Keep both as separate observations (user × window); never de-duplicate. User-level questions use a separate derived table with a stated rule | 2.3, 2.4 |
| Test rosters | `sample_submission_zero.csv`: 970,960 users, placeholder predictions; documented original March test set | `sample_submission_v2.csv`: 907,471 users, placeholder predictions; documented April test set, labels not public | `sub_zero` has exactly the `train_v2` users; 88.32% of `sub_v2` users are in `train_v2`, 88.26% in `train_v1` | Not combined; rosters only, no evaluation labels | 2.5 |
| Members | `members_v3.csv` (single release): 6,769,473 members, unique `msno` | n/a | 88.34% of `train_v1` users and 88.67% of `train_v2` users have a member profile | Left join; keep users without a profile | 3.1, 3.5, 3.6 |
| Transactions | `transactions.csv`: 21,547,746 rows, 2,363,626 users, 2015-01-01 to 2017-02-28 | `transactions_v2.csv`: 1,431,009 rows, 1,197,050 users; 1,069,822 rows in March 2017 and 361,187 before | No identical rows; 94.78% of v2 users also in v1; 7,249 (user, date) keys occur in both releases, 89.59% of their row pairs differ only in expiration date; the pre-March v2 rows look like a supplement to v1 (*hypothesis*) | Stack as `tx_all` with a `release` column and keep every row; define the handling of shared keys when membership sequences are rebuilt | 4.7, 4.7.1 |
| User logs | `user_logs.csv`: 392,106,543 rows, 5,234,111 users, 2015-01-01 to 2017-02-28 | `user_logs_v2.csv`: 18,396,362 rows, 1,103,894 users, 2017-03-01 to 2017-03-31 | No date occurs in both releases; (`msno`, `date`) is unique within each; 998,583 users in both (90.46% of v2 users) | Stack as `logs_all` with a `release` column; no key can repeat | 5.2, 5.5 |

**Reading the table:** the logs are a time extension (v2 adds March only); the transactions are an extension plus a small supplement of earlier rows; the two label tables describe different windows of mostly the same users. In the audit phase the releases are only combined through views; building physical merged tables belongs to the staging step after the combination rules are settled.


## 2. Training and test-user files

### 2.1 Label tables and test rosters — purpose and schema

`train_v1` (`train.csv`) and `train_v2` (`train_v2.csv`) hold one supplied binary outcome per user: 
- `msno` identifies the user and `is_churn` is the label (`1` churn, `0` renewal). 
- `sub_zero` and `sub_v2` are test-user rosters with placeholder predictions. 
- Label proportions describe each labeled population, not the entire subscriber base, and the two label files describe different windows, so their rates are never pooled. 
- The windows are documented (Section 1.2) but not yet verified from the local transactions.

### 2.2 Structure, key uniqueness, and completeness

In [35]:
train_tables = ["train_v1", "train_v2"]

show("Structure: rows, columns, distinct users",
     pd.concat({t: ac.table_summary(con, t, key="msno")["value"] for t in train_tables}, axis=1))
show("Key uniqueness: `msno` must identify one row",
     pd.concat({t: ac.key_uniqueness(con, t, "msno")["value"] for t in train_tables}, axis=1),
     note="excess_rows counts rows beyond the first for a repeated key; 0 means the key is unique.")
show("Missing values per column (NULL or blank)",
     pd.concat({t: ac.missingness(con, t)["missing"] for t in train_tables}, axis=1))

**Structure: rows, columns, distinct users**

,train_v1,train_v2
rows,"992,931","970,960"
columns,2,2
unique_msno,"992,931","970,960"


**Key uniqueness: `msno` must identify one row**  
*excess_rows counts rows beyond the first for a repeated key; 0 means the key is unique.*

,train_v1,train_v2
rows,"992,931","970,960"
missing_key_rows,0,0
unique_keys,"992,931","970,960"
duplicated_keys,0,0
excess_rows,0,0
rows_in_duplicated_keys,0,0
max_rows_per_key,1,1


**Missing values per column (NULL or blank)**

,train_v1,train_v2
msno,0,0
is_churn,0,0


### 2.3 Label values and distribution

Counts and percentages are within each labeled population, not platform churn rates.

In [36]:
labels = (ac.value_counts_pct(con, "train_v1", "is_churn").set_index("value")
          .join(ac.value_counts_pct(con, "train_v2", "is_churn").set_index("value"),
                lsuffix="_v1", rsuffix="_v2")
          .rename_axis("is_churn"))
show("Label distribution (`is_churn`) in each label table", labels,
     note="Percentages are within each labeled population, not platform churn rates.")

**Label distribution (`is_churn`) in each label table**  
*Percentages are within each labeled population, not platform churn rates.*

,count_v1,percentage_v1,count_v2,percentage_v2
is_churn,,,,
0,"929,460",93.61%,"883,630",91.01%
1,"63,471",6.39%,"87,330",8.99%


### 2.4 Overlap between the two label tables

- The same user can appear in both windows. 
- Each row is a user × window observation; the transition table shows how the label changes for users present in both. 
- Overlap alone is not leakage, but it affects how validation is split.

In [37]:
transitions = con.sql("""
    SELECT a.is_churn AS label_v1, b.is_churn AS label_v2, count(*) AS users
    FROM train_v1 a JOIN train_v2 b USING (msno)
    GROUP BY ALL ORDER BY ALL
""").df().pivot(index="label_v1", columns="label_v2", values="users")
transitions.index = [f"v1 label = {i}" for i in transitions.index]
transitions.columns = [f"v2 label = {c}" for c in transitions.columns]

show("Label transitions for users present in both tables (users)", transitions,
     note="Rows: label in train_v1 (February window, documented). Columns: label in train_v2 (March window).")
show("Coverage: share of train_v1 users also in train_v2", ac.coverage(con, "train_v1", "train_v2", "msno"))
show("Coverage: share of train_v2 users also in train_v1", ac.coverage(con, "train_v2", "train_v1", "msno"))

**Label transitions for users present in both tables (users)**  
*Rows: label in train_v1 (February window, documented). Columns: label in train_v2 (March window).*

,v2 label = 0,v2 label = 1
v1 label = 0,"824,659","40,721"
v1 label = 1,"5,269","11,052"


**Coverage: share of train_v1 users also in train_v2**

,value
left_keys,"992,931"
matched,"881,701"
unmatched,"111,230"
matched_percentage,88.80%


**Coverage: share of train_v2 users also in train_v1**

,value
left_keys,"970,960"
matched,"881,701"
unmatched,"89,259"
matched_percentage,90.81%


### 2.5 Test-user rosters — `sample_submission_zero` and `sample_submission_v2`

**Purpose:** Test-user IDs and the prediction-submission format. `is_churn` holds placeholder predictions, not observed outcomes. 

Documented roles: 
- `sub_zero` is the original March 2017 test set (its labels were later released as `train_v2`)；
- `sub_v2` is the April 2017 test set, whose true labels are not public.

#### 2.5.1 Structure, completeness, user key, and placeholder values

In [38]:
test_tables = ["sub_zero", "sub_v2"]

show("Structure: rows, columns, distinct users",
     pd.concat({t: ac.table_summary(con, t, key="msno")["value"] for t in test_tables}, axis=1))
show("Key uniqueness: `msno` must identify one row",
     pd.concat({t: ac.key_uniqueness(con, t, "msno")["value"] for t in test_tables}, axis=1))
for t in test_tables:
    show(f"Placeholder prediction values in {t}", ac.value_counts_pct(con, t, "is_churn"),
         note="Template values only; this is not a churn-rate calculation.")

**Structure: rows, columns, distinct users**

,sub_zero,sub_v2
rows,"970,960","907,471"
columns,2,2
unique_msno,"970,960","907,471"


**Key uniqueness: `msno` must identify one row**

,sub_zero,sub_v2
rows,"970,960","907,471"
missing_key_rows,0,0
unique_keys,"970,960","907,471"
duplicated_keys,0,0
excess_rows,0,0
rows_in_duplicated_keys,0,0
max_rows_per_key,1,1


**Placeholder prediction values in sub_zero**  
*Template values only; this is not a churn-rate calculation.*

,value,count,percentage
0,0,"970,960",100.00%


**Placeholder prediction values in sub_v2**  
*Template values only; this is not a churn-rate calculation.*

,value,count,percentage
0,0,"907,471",100.00%


#### 2.5.2 Overlap of the test rosters with the label tables

The denominator is each roster's distinct users. Shared IDs do not by themselves establish leakage; label windows and feature cutoffs must be verified.

In [39]:
pairs = [("sub_zero", "train_v2"), ("sub_v2", "train_v2"), ("sub_v2", "train_v1"), ("sub_v2", "sub_zero")]
coverage_table = pd.DataFrame(
    {f"{a} in {b}": ac.coverage(con, a, b, "msno")["value"] for a, b in pairs}).T
show("Coverage: share of each roster's users found in a reference table", coverage_table)

**Coverage: share of each roster's users found in a reference table**

,left_keys,matched,unmatched,matched_percentage
sub_zero in train_v2,"970,960","970,960",0,100.00%
sub_v2 in train_v2,"907,471","801,490","105,981",88.32%
sub_v2 in train_v1,"907,471","800,937","106,534",88.26%
sub_v2 in sub_zero,"907,471","801,490","105,981",88.32%


### 2.6 Findings, impact, and decisions

| Finding | Evidence (full data) | Impact / decision |
|---|---|---|
| Label tables are clean | `train_v1`: 992,931 rows; `train_v2`: 970,960 rows. `msno` unique and complete; `is_churn` binary with no missing values | One observation per user in each table |
| Label rates differ by window | 6.39% (63,471 of 992,931) in `train_v1`; 8.99% (87,330 of 970,960) in `train_v2` | Rates describe each labeled population, not the platform; the two windows are never pooled |
| Most users appear in both windows | 881,701 users in both (88.80% of `train_v1`, 90.81% of `train_v2`). Transitions v1 → v2: 0→0 824,659; 0→1 40,721; 1→0 5,269; 1→1 11,052 | Keep both observations (user × window). User-level questions use a separate derived table with a stated rule; raw label tables are never de-duplicated. Whether to restrict analysis to overlap users is deferred: default is all users per window, with overlap and non-overlap users reported separately |
| Test rosters are templates | `sub_v2`: 907,471 unique users, all placeholder 0; 88.32% in `train_v2`, 88.26% in `train_v1`. `sub_zero`: 970,960 unique users, all placeholder 0 | Rosters only; no test labels exist here, so `sub_v2` cannot be used for evaluation |
| `sub_zero` equals the `train_v2` user set | All 970,960 `sub_zero` users are in `train_v2` (100% match) | Consistent with the official description (original March test set, labels later released as `train_v2`) |
| Windows are documented, not verified | Documented: `train` = February 2017 expirations, `train_v2` = March 2017 | Verify from transactions. A v1 label is not used as a v2 feature until the windows are verified |

**Not checked here:** Member, transaction, and log coverage of these users; label-window verification; label reconstruction from transactions.

## 3. Members — `members_v3.csv`

**Purpose:** Member profile attributes. **Observed grain:** one row per `msno` (verified in 3.1).

**Source fields:** `msno`, `city`, `bd`, `gender`, `registered_via`, `registration_init_time`.


### 3.1 Structure, key uniqueness, and completeness


In [40]:
show("Structure: rows, columns, distinct users", ac.table_summary(con, "members", key="msno"))
show("Key uniqueness: `msno` must identify one row", ac.key_uniqueness(con, "members", "msno"),
     note="excess_rows counts rows beyond the first for a repeated key; 0 means the key is unique.")
show("Missing values per column (NULL or blank)", ac.missingness(con, "members"))
show("Preview: first five rows as stored (user identifier omitted)",
     con.sql("SELECT city, bd, gender, registered_via, registration_init_time FROM members LIMIT 5").df())

**Structure: rows, columns, distinct users**

,value
rows,"6,769,473"
columns,6
unique_msno,"6,769,473"


**Key uniqueness: `msno` must identify one row**  
*excess_rows counts rows beyond the first for a repeated key; 0 means the key is unique.*

,value
rows,"6,769,473"
missing_key_rows,0
unique_keys,"6,769,473"
duplicated_keys,0
excess_rows,0
rows_in_duplicated_keys,0
max_rows_per_key,1


**Missing values per column (NULL or blank)**

,missing,percentage
msno,0,0.00%
city,0,0.00%
bd,0,0.00%
gender,"4,429,505",65.43%
registered_via,0,0.00%
registration_init_time,0,0.00%


**Preview: first five rows as stored (user identifier omitted)**

,city,bd,gender,registered_via,registration_init_time
0,1,0,–,11,20110911
1,1,0,–,7,20110914
2,1,0,–,11,20110915
3,1,0,–,11,20110915
4,6,32,female,9,20110915


### 3.2 Age (`bd`) and gender validity

**Purpose:** size the missing-gender and implausible-age problems before any demographic analysis. 
- Zero age is a review flag, not a confirmed missing-value definition; 
- ages outside 1–100 need review. 

**Logic:** `numeric_range` reports the parseable range of `bd`. `flag_counts` counts rows for each named condition against all rows; the conditions use `TRY_CAST`, so a non-numeric value would be counted as none of them (and would show in `non_numeric` above). The last flag measures how often missing gender and zero age occur together. The two value tables show the most frequent ages and the most frequent out-of-range ages.

**What to look for:** 
- how large `age_zero` is relative to all rows;
- whether `age_1_to_100` is a small minority; 
- extreme positive or negative values; 
- whether missing gender & zero age mostly coincide (suggesting incomplete profiles, not a proven cause).

In [41]:
AGE = 'TRY_CAST("bd" AS DOUBLE)'
GENDER_MISSING = "(gender IS NULL OR trim(gender) = '')"

show("Numeric range of `bd` (age)", ac.numeric_range(con, "members", ["bd"]).T)
show("Age and gender review flags", ac.flag_counts(con, "members", {
    "age_zero": f"{AGE} = 0",
    "age_negative": f"{AGE} < 0",
    "age_1_to_100": f"{AGE} BETWEEN 1 AND 100",
    "age_above_100": f"{AGE} > 100",
    "gender_missing": GENDER_MISSING,
    "gender_missing_and_age_zero": f"{GENDER_MISSING} AND {AGE} = 0",
}), note="Flags overlap (not a partition). Percentages are of all member rows.")
show("Ten most frequent `bd` values", ac.value_counts_pct(con, "members", "bd", limit=10))
show("Twenty most frequent out-of-range ages (negative or above 100)", con.sql(f"""
    SELECT bd AS value, count(*) AS count
    FROM members WHERE {AGE} < 0 OR {AGE} > 100
    GROUP BY bd ORDER BY count DESC, bd LIMIT 20
""").df())

**Numeric range of `bd` (age)**

,bd
missing,0
non_numeric,0
negative,274
zero,"4,540,215"
min,"-7,168"
max,"2,016"


**Age and gender review flags**  
*Flags overlap (not a partition). Percentages are of all member rows.*

,rows,percentage_of_all_rows
age_zero,"4,540,215",67.07%
age_negative,274,<0.01%
age_1_to_100,"2,223,607",32.85%
age_above_100,"5,377",0.08%
gender_missing,"4,429,505",65.43%
gender_missing_and_age_zero,"4,388,613",64.83%


**Ten most frequent `bd` values**

,value,count,percentage
0,0,"4,540,215",67.07%
1,22,"112,200",1.66%
2,21,"110,574",1.63%
3,20,"110,452",1.63%
4,27,"102,769",1.52%
5,23,"101,500",1.50%
6,24,"97,252",1.44%
7,26,"92,433",1.37%
8,25,"91,514",1.35%
9,19,"91,374",1.35%


**Twenty most frequent out-of-range ages (negative or above 100)**

,value,count
0,112,"1,309"
1,106,743
2,103,449
3,117,439
4,102,424
5,104,307
6,101,291
7,107,290
8,111,257
9,105,205


### 3.3 Registration dates

**Purpose:** verify that `registration_init_time` is a real `YYYYMMDD` date and see how registrations are spread over time.

**Logic:** `date_validity` counts missing and invalid values (not a real calendar date) and returns the earliest and latest valid date. The year table counts members per registration year. The two cutoff flags are informational: they count members who registered after the last day covered by each release's logs and transactions, because they cannot have log or transaction history inside that release. They do not fix an analysis cutoff.

**What to look for:** 
- zero missing and invalid dates; 
- a latest date that is later than the end of the data window (a reminder to apply a cutoff in feature work); 
- a sensible year distribution.

In [42]:
show("Registration date validity and range", ac.date_validity(con, "members", ["registration_init_time"]).T)
show("Members by registration year", con.sql("""
    SELECT substr(registration_init_time, 1, 4) AS year, count(*) AS members,
           round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS percentage
    FROM members GROUP BY year ORDER BY year
""").df())
show("Registrations after the end of each release window (informational)", ac.flag_counts(con, "members", {
    "registered_after_20170228": "registration_init_time > '20170228'",
    "registered_after_20170331": "registration_init_time > '20170331'",
}), note="Compact YYYYMMDD strings compare correctly as text. 2017-02-28 ends v1 logs and transactions; 2017-03-31 ends v2 logs.")

**Registration date validity and range**

,registration_init_time
missing,0
invalid,0
earliest,2004-03-26
latest,2017-04-29


**Members by registration year**

,year,members,percentage
0,2004,"26,234",0.39%
1,2005,"41,349",0.61%
2,2006,"53,953",0.80%
3,2007,"89,830",1.33%
4,2008,"67,690",1.00%
5,2009,"63,633",0.94%
6,2010,"115,075",1.70%
7,2011,"179,051",2.64%
8,2012,"283,190",4.18%
9,2013,"524,722",7.75%


**Registrations after the end of each release window (informational)**  
*Compact YYYYMMDD strings compare correctly as text. 2017-02-28 ends v1 logs and transactions; 2017-03-31 ends v2 logs.*

,rows,percentage_of_all_rows
registered_after_20170228,"154,166",2.28%
registered_after_20170331,"55,094",0.81%


### 3.4 Category codes and the unusual registration channel

**Purpose:** list every value of `gender`, `city`, and `registered_via`, and inspect the one `registered_via = -1` record. Code meanings are not inferred from their numbers.

**Logic:** `value_counts_pct` groups by value including missing values (shown as –), most frequent first. The final table pulls the rows whose channel is `-1`.

**What to look for:** how concentrated each column is (for example, one dominant city); whether rare codes exist; whether missing gender is the largest group; and whether the `-1` record looks different from other members.

In [43]:
for column in ["gender", "city", "registered_via"]:
    show(f"Value distribution: `{column}`", ac.value_counts_pct(con, "members", column))
show("Records with `registered_via = -1`", con.sql("""
    SELECT city, bd, gender, registered_via, registration_init_time
    FROM members WHERE TRY_CAST(registered_via AS INTEGER) = -1
""").df())

**Value distribution: `gender`**

,value,count,percentage
0,–,"4,429,505",65.43%
1,male,"1,195,355",17.66%
2,female,"1,144,613",16.91%


**Value distribution: `city`**

,value,count,percentage
0,1,"4,804,326",70.97%
1,5,"385,069",5.69%
2,13,"320,978",4.74%
3,4,"246,848",3.65%
4,22,"210,407",3.11%
5,15,"190,213",2.81%
6,6,"135,200",2.00%
7,14,"89,940",1.33%
8,12,"66,843",0.99%
9,9,"47,639",0.70%


**Value distribution: `registered_via`**

,value,count,percentage
0,4,"2,793,213",41.26%
1,3,"1,643,208",24.27%
2,9,"1,482,863",21.91%
3,7,"805,895",11.90%
4,11,"25,047",0.37%
5,13,"5,455",0.08%
6,8,"3,982",0.06%
7,5,"3,115",0.05%
8,17,"1,494",0.02%
9,2,"1,452",0.02%


**Records with `registered_via = -1`**

,city,bd,gender,registered_via,registration_init_time
0,1,0,–,-1,20140625


### 3.5 Coverage: label tables and test rosters to members

**Purpose:** measure how many labeled and test users have a member profile. The denominator is the distinct users of the left table; matching users are those whose `msno` appears in `members`.

**Logic:** `coverage` left-joins the distinct users of each table to the distinct member keys and reports matched and unmatched counts.

**What to look for:** the unmatched share for each table, and whether it is similar across `train_v1`, `train_v2`, and the two test rosters. A large unmatched share means profile features will be absent for those users.

In [44]:
coverage_tables = ["train_v1", "train_v2", "sub_zero", "sub_v2"]

show("Coverage: share of each table's users found in `members`",
     pd.concat({t: ac.coverage(con, t, "members", "msno")["value"] for t in coverage_tables}, axis=1))

**Coverage: share of each table's users found in `members`**

,train_v1,train_v2,sub_zero,sub_v2
left_keys,"992,931","970,960","970,960","907,471"
matched,"877,161","860,967","860,967","795,090"
unmatched,"115,770","109,993","109,993","112,381"
matched_percentage,88.34%,88.67%,88.67%,87.62%


### 3.6 Preserve the labeled population with a left join

**Purpose:** decide whether users without a member profile can be dropped. They cannot be dropped without changing the population: this step compares label rates for matched and unmatched users and creates the joined views used later.

**Logic:** each label table is left-joined to `members` on `msno` in a view (`train_v1_members`, `train_v2_members`) with a `has_member_profile` flag. The row-preservation table checks that the join neither adds nor removes label rows (valid only because `msno` is unique in `members`, see 3.1). The label table groups users by that flag and reports the churn-label rate in each group.

**What to look for:** `rows_after_join` equal to `rows_before_join`; the difference between matched and unmatched label rates. This comparison describes the labeled populations only and does not establish a causal effect of profile completeness on churn.

In [45]:
profile_columns = "m.city, m.bd, m.gender, m.registered_via, m.registration_init_time"
for t in ["train_v1", "train_v2"]:
    con.execute(f"""
        CREATE OR REPLACE VIEW {t}_members AS
        SELECT t.*, {profile_columns}, m.msno IS NOT NULL AS has_member_profile
        FROM {t} t LEFT JOIN members m ON t.msno = m.msno
    """)

preservation = pd.DataFrame({
    t: {"rows_before_join": ac.table_summary(con, t)["value"]["rows"],
        "rows_after_join": ac.table_summary(con, f"{t}_members")["value"]["rows"]}
    for t in ["train_v1", "train_v2"]})
show("Row preservation: left join of each label table to `members`", preservation,
     note="Equal counts confirm no label row was added or removed.")

label_by_profile = pd.concat({t: con.sql(f"""
    SELECT has_member_profile, count(*) AS users,
           count(*) FILTER (WHERE TRY_CAST(is_churn AS INTEGER) = 1) AS churn_labels,
           100.0 * count(*) FILTER (WHERE TRY_CAST(is_churn AS INTEGER) = 1) / count(*) AS churn_label_percentage
    FROM {t}_members GROUP BY has_member_profile ORDER BY has_member_profile
""").df().set_index("has_member_profile") for t in ["train_v1", "train_v2"]})
show("Churn-label rate by member-profile availability", label_by_profile,
     note="Rates are within each labeled population. Dropping unmatched users would change the population and its label rate.")

**Row preservation: left join of each label table to `members`**  
*Equal counts confirm no label row was added or removed.*

,train_v1,train_v2
rows_before_join,"992,931","970,960"
rows_after_join,"992,931","970,960"


**Churn-label rate by member-profile availability**  
*Rates are within each labeled population. Dropping unmatched users would change the population and its label rate.*

### 3.7 Findings, impact, and decisions

Counts below were reproduced by a read-only run of the cells above on the full `members_v3.csv`, `train.csv`, `train_v2.csv`, and both sample submissions; re-run the notebook to confirm them in your kernel.

| Area | Verified finding (full data) | Analytical impact | Decision |
|---|---|---|---|
| Structure and key | 6,769,473 rows, 6 source columns; no missing or duplicate `msno` | Supports a one-to-one member lookup | Use `msno` as the member key |
| Gender | 4,429,505 missing (65.43%); non-missing values are `male` and `female` | Complete-profile analysis would exclude most members | Preserve missingness; keep an explicit unknown group |
| Age (`bd`) | 4,540,215 zeros (67.07%); 274 negative; 5,377 above 100; range -7,168 to 2,016; only 2,223,607 (32.85%) within 1–100 | Raw age averages and segments are unreliable | Preserve `bd`; confirm the meaning of zero and define review flags before any cleaning |
| Overlapping gaps | 4,388,613 rows (64.83%) have both missing gender and zero age | Suggests incomplete profiles, not a confirmed cause | Do not impute from this association alone |
| Registration dates | No missing or invalid dates; 2004-03-26 to 2017-04-29; 154,166 members (2.28%) registered after 2017-02-28 and 55,094 (0.81%) after 2017-03-31 | Some registrations fall after a release window and cannot have history in it | Apply a verified cutoff in feature work |
| Category codes | `city` and `registered_via` have no missing values; city 1 holds 70.97%; channel `-1` occurs once | Code semantics remain unknown | Preserve codes; do not guess meanings |
| Label-table coverage | Matched to `members`: `train_v1` 877,161 of 992,931 (88.34%); `train_v2` 860,967 of 970,960 (88.67%) | Profile features are absent for about 11–12% of labeled users | Keep all labeled users with a left join |
| Test-roster coverage | `sub_zero` 88.67% (same users as `train_v2`); `sub_v2` 795,090 of 907,471 (87.62%) | Same gap at scoring time | Handle missing profiles explicitly in any model |
| Effect of an inner join | Unmatched users have a lower label rate: `train_v2` 5.35% vs 9.46% matched; `train_v1` 5.02% vs 6.57% | Dropping them would change the population and its label rate (`train_v2` 8.99% to 9.46%) | Left join preserves all 992,931 and 970,960 labeled rows (`train_v1_members`, `train_v2_members`) |

**Open:** meaning of zero age and of channel `-1`; treatment rules for demographics; temporal eligibility of members against the cutoff. The comparison of label rates is descriptive and does not show that profile completeness affects churn. No source values have been replaced.

## 4. Transactions — `transactions.csv` (v1) and `transactions_v2.csv` (v2)

**Purpose:** Subscription and payment records. 

**Grain:** a source transaction record; multiple rows per member are expected. No unique transaction identifier has been established.

**Source fields:** `msno`, `payment_method_id`, `payment_plan_days`, `plan_list_price`, `actual_amount_paid`, `is_auto_renew`, `transaction_date`, `membership_expire_date`, `is_cancel`.

**Method:** both releases are checked separately and side by side (`tx_v1`, `tx_v2`), then compared in 4.7. The merged view `tx_all` (a `release` column plus the nine fields) is used only where one query must cover both. All values stay text in the registered tables; each check interprets numbers and dates itself, so source values are never altered. The CSVs are large (v1 is about 1.7 GB), so some cells take a while.

### 4.1 Structure, completeness, and candidate keys

**Purpose:** confirm the shape of each release, that no source value is missing, and what a row can and cannot be identified by.

**Logic:** `table_summary` counts rows, columns, and distinct users. `missingness` counts NULL or blank values per column. `exact_duplicates` compares all nine fields. `key_uniqueness` on (`msno`, `transaction_date`) measures same-user same-day rows; those are not automatically duplicates (a member can legitimately transact twice in a day), so this is a grain check, not a de-duplication rule.

**What to look for:** zero missing values; zero exact duplicates; and a nonzero count of repeated (`msno`, `transaction_date`) keys, which shows that this pair is not a unique transaction key.

In [46]:
TX = ["tx_v1", "tx_v2"]

show("Structure: rows, columns, distinct users",
     pd.concat({t: ac.table_summary(con, t, key="msno")["value"] for t in TX}, axis=1))
show("Missing values per column (NULL or blank)",
     pd.concat({t: ac.missingness(con, t)["missing"] for t in TX}, axis=1))
show("Exact duplicate rows (all nine source fields)",
     pd.concat({t: ac.exact_duplicates(con, t)["value"] for t in TX}, axis=1),
     note="excess_rows counts rows beyond the first among identical rows; 0 means no exact duplicates.")
show("Candidate key (`msno`, `transaction_date`): same-user same-day rows",
     pd.concat({t: ac.key_uniqueness(con, t, ["msno", "transaction_date"])["value"] for t in TX}, axis=1),
     note="Repeated keys are not automatically duplicates; this pair is not a unique transaction key.")

**Structure: rows, columns, distinct users**

,tx_v1,tx_v2
rows,"21,547,746","1,431,009"
columns,9,9
unique_msno,"2,363,626","1,197,050"


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Missing values per column (NULL or blank)**

,tx_v1,tx_v2
msno,0,0
payment_method_id,0,0
payment_plan_days,0,0
plan_list_price,0,0
actual_amount_paid,0,0
is_auto_renew,0,0
transaction_date,0,0
membership_expire_date,0,0
is_cancel,0,0


...

**Exact duplicate rows (all nine source fields)**  
*excess_rows counts rows beyond the first among identical rows; 0 means no exact duplicates.*

,tx_v1,tx_v2
rows,"21,547,746","1,431,009"
distinct_rows,"21,544,407","1,431,009"
excess_rows,"3,339",0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Candidate key (`msno`, `transaction_date`): same-user same-day rows**  
*Repeated keys are not automatically duplicates; this pair is not a unique transaction key.*

,tx_v1,tx_v2
rows,"21,547,746","1,431,009"
missing_key_rows,0,0
unique_keys,"21,261,342","1,397,717"
duplicated_keys,"256,003","27,942"
excess_rows,"286,404","33,292"
rows_in_duplicated_keys,"542,407","61,234"
max_rows_per_key,48,131


### 4.2 Numeric ranges, flags, and plan duration

**Purpose:** check that numeric fields parse, that the two flags are binary, and what plan durations and prices look like. Payment-method codes are labels, not continuous measures, and cancellation flags are not churn labels.

**Logic:** `numeric_range` reports missing, non-numeric, negative, and zero counts and min/max per column. `flag_counts` counts rows meeting each condition against all rows of that release. The helper `by_release` puts the value distribution of a column for both releases side by side (a dash means the value does not occur in that release).

**What to look for:** `non_numeric = 0`; flag values only 0 and 1; zero-valued plan days and prices (examined further in 4.6); how the plan-duration mix differs between releases.

In [47]:
NUM = ["payment_method_id", "payment_plan_days", "plan_list_price", "actual_amount_paid"]
def n(column):
    return f'TRY_CAST("{column}" AS DOUBLE)'

def by_release(column, limit=None):
    parts = [ac.value_counts_pct(con, t, column, limit).set_index("value").add_suffix(f"_{r}")
             for t, r in zip(TX, ["v1", "v2"])]
    return parts[0].join(parts[1], how="outer").sort_values("count_v1", ascending=False, na_position="last")

for t in TX:
    show(f"Numeric range in {t}", ac.numeric_range(con, t, NUM).T)
show("Flag and price review flags", pd.concat({t: ac.flag_counts(con, t, {
    "auto_renew_not_binary": "coalesce(is_auto_renew NOT IN ('0', '1'), true)",
    "cancel_not_binary": "coalesce(is_cancel NOT IN ('0', '1'), true)",
    "plan_days_zero": f"{n('payment_plan_days')} = 0",
    "list_price_zero": f"{n('plan_list_price')} = 0",
    "paid_zero": f"{n('actual_amount_paid')} = 0",
    "paid_differs_from_list": f"{n('actual_amount_paid')} <> {n('plan_list_price')}",
}) for t in TX}, axis=1), note="Flags overlap (not a partition). Percentages are of all rows of each release.")
show("`is_auto_renew` by release", by_release("is_auto_renew"))
show("`is_cancel` by release", by_release("is_cancel"))
show("`payment_plan_days` by release (top 15 by v1 count)", by_release("payment_plan_days").head(15))

**Numeric range in tx_v1**

,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid
missing,0,0,0,0
non_numeric,0,0,0,0
negative,0,0,0,0
zero,0,"870,124","1,498,544","1,196,876"
min,1,0,0,0
max,41,450,"2,000","2,000"


**Numeric range in tx_v2**

,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid
missing,0,0,0,0
non_numeric,0,0,0,0
negative,0,0,0,0
zero,0,"2,218","18,413","21,448"
min,2,0,0,0
max,41,450,"2,000","2,000"


**Flag and price review flags**  
*Flags overlap (not a partition). Percentages are of all rows of each release.*

...

,count_v1,percentage_v1,count_v2,percentage_v2
value,,,,
1,"18,357,950",85.20%,"1,123,775",78.53%
0,"3,189,796",14.80%,"307,234",21.47%


**`is_cancel` by release**

,count_v1,percentage_v1,count_v2,percentage_v2
value,,,,
0,"20,690,895",96.02%,"1,395,876",97.54%
1,"856,851",3.98%,"35,133",2.46%


**`payment_plan_days` by release (top 15 by v1 count)**

,count_v1,percentage_v1,count_v2,percentage_v2
value,,,,
30,"18,956,290",87.97%,"1,217,998",85.11%
0,"870,124",4.04%,"2,218",0.15%
31,"766,608",3.56%,4,<0.01%
7,"577,639",2.68%,"12,168",0.85%
195,"110,234",0.51%,"28,568",2.00%
410,"80,139",0.37%,"82,097",5.74%
180,"52,272",0.24%,"23,900",1.67%
10,"38,216",0.18%,416,0.03%
100,"24,154",0.11%,"4,098",0.29%


### 4.3 Dates, coverage, and early or distant dates

**Purpose:** verify that both date fields are real `YYYYMMDD` dates, see which months each release covers, and size the rows that fall outside the documented window or look implausible (expiration dates in the distant past or future).

**Logic:** `date_validity` reports missing and invalid values and the range of each date column. The monthly table counts transaction rows per calendar month in each release. The flag table counts rows by early or late dates; compact `YYYYMMDD` strings compare correctly as text, and dates were verified valid first. The last table lists the most frequent very early expiration dates in v1.

**What to look for:** no invalid dates; whether v1 ends at 2017-02-28 and v2 starts at 2017-03-01 as documented, or whether v2 also carries earlier transactions (this notebook measures it); expiration dates before 2015 or far in the future, which need a rule before any date arithmetic.

In [48]:
show("Date validity and range", pd.concat({t: ac.date_validity(con, t, ["transaction_date", "membership_expire_date"]) for t in TX}))
show("Transaction rows per month and release", con.sql("""
    SELECT substr(transaction_date, 1, 4) || '-' || substr(transaction_date, 5, 2) AS month,
           count(*) FILTER (WHERE release = 'v1') AS rows_v1,
           count(*) FILTER (WHERE release = 'v2') AS rows_v2
    FROM tx_all GROUP BY month ORDER BY month
""").df().set_index("month"), max_rows=40,
     note="A month appearing in both columns means the two releases overlap in time.")
show("Early and late date flags", pd.concat({t: ac.flag_counts(con, t, {
    "transaction_before_20170301": "transaction_date < '20170301'",
    "expiry_before_20150101": "membership_expire_date < '20150101'",
    "expiry_in_1970": "membership_expire_date < '19710101'",
    "expiry_after_20180101": "membership_expire_date > '20180101'",
    "expiry_after_20200101": "membership_expire_date > '20200101'",
}) for t in TX}, axis=1), note="Informational thresholds, not cleaning rules.")
show("Most frequent expiration dates before 2015 in v1", con.sql("""
    SELECT membership_expire_date AS value, count(*) AS count FROM tx_v1
    WHERE membership_expire_date < '20150101' GROUP BY 1 ORDER BY count DESC, 1 LIMIT 10
""").df())

**Date validity and range**

**Transaction rows per month and release**  
*A month appearing in both columns means the two releases overlap in time.*

,rows_v1,rows_v2
month,,
2015-01,"576,349","1,169"
2015-02,"569,421","1,504"
2015-03,"657,379","3,371"
2015-04,"595,444","3,579"
2015-05,"601,063","4,393"
2015-06,"818,903","5,179"
2015-07,"697,086","6,247"
2015-08,"740,477","7,095"
2015-09,"744,636","7,692"


**Early and late date flags**  
*Informational thresholds, not cleaning rules.*

**Most frequent expiration dates before 2015 in v1**

,value,count
0,19700101,"1,776"
1,20141231,393
2,20141224,122
3,20141227,112
4,20131226,102
5,20141223,99
6,20141225,99
7,20141226,94
8,20141125,63
9,20130315,60


### 4.4 Expiration before transaction date

**Purpose:** find rows whose `membership_expire_date` is earlier than `transaction_date`, and see how they relate to cancellations.

**Logic:** the first table counts such rows within each release and `is_cancel` value (a cancellation row can legitimately carry an earlier expiration). The second lists up to 20 non-cancellation rows with this pattern, because those are the unexplained ones.

**What to look for:** whether the anomalies are almost all cancellation rows; the size of the non-cancellation remainder and what those rows have in common (plan days, prices). Rows are preserved; the cause is not established here.

In [49]:
show("Expiration before transaction date, by release and cancellation flag", con.sql("""
    SELECT release, is_cancel, count(*) AS rows,
           count(*) FILTER (WHERE membership_expire_date < transaction_date) AS expiry_before_transaction,
           100.0 * count(*) FILTER (WHERE membership_expire_date < transaction_date) / count(*) AS anomaly_percentage
    FROM tx_all GROUP BY ALL ORDER BY ALL
""").df().set_index(["release", "is_cancel"]))
show("Non-cancellation rows that expire before the transaction date (up to 20)", con.sql("""
    SELECT release, payment_plan_days, plan_list_price, actual_amount_paid, is_auto_renew,
           transaction_date, membership_expire_date
    FROM tx_all WHERE membership_expire_date < transaction_date AND is_cancel = '0'
    ORDER BY release, transaction_date LIMIT 20
""").df())

**Expiration before transaction date, by release and cancellation flag**

**Non-cancellation rows that expire before the transaction date (up to 20)**

,release,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date
0,v1,30,149,149,1,20150102,20141017
1,v1,1,0,0,0,20150106,20150105
2,v1,1,0,0,0,20150109,20150107
3,v1,30,149,149,1,20150110,20130120
4,v1,30,149,149,1,20150110,20121128
5,v1,30,149,149,1,20150111,20130413
6,v1,1,0,0,0,20150112,20150110
7,v1,30,149,149,1,20150114,20141115
8,v1,30,149,149,1,20150114,20141114
9,v1,30,149,149,1,20150114,20141115


### 4.5 Long-expiry records and one accumulation case study

**Purpose:** see how far expiration dates extend beyond the transaction date, and test one extreme member as a case study. The case study is a targeted investigation, not a population-wide validation of how subscriptions accumulate.

**Logic:** `days_to_expiry` is the calendar-day gap between the two dates. The first tables list the ten largest gaps per release (members shown by a short prefix) and count rows above fixed gap thresholds. For the member with the single largest gap, all records from both releases are sorted by transaction date then expiration date, and each expiration increment is compared with that row's `payment_plan_days`.

**What to look for:** whether the longest gaps belong to few members; whether increments match plan days (consistent with stacked renewals), and where the few mismatches occur; a distant expiration date alone is not a reason to delete a record.

In [50]:
DAYS = "date_diff('day', TRY_STRPTIME(transaction_date, '%Y%m%d'), TRY_STRPTIME(membership_expire_date, '%Y%m%d'))"

for t in TX:
    show(f"Ten largest `days_to_expiry` in {t}", con.sql(f"""
        SELECT left(msno, 8) AS member, payment_plan_days, actual_amount_paid, transaction_date,
               membership_expire_date, {DAYS} AS days_to_expiry, is_cancel
        FROM {t} ORDER BY days_to_expiry DESC NULLS LAST LIMIT 10
    """).df())
show("Rows above fixed `days_to_expiry` thresholds", pd.concat({t: ac.flag_counts(con, t, {
    "gap_over_450_days": f"{DAYS} > 450",
    "gap_over_1000_days": f"{DAYS} > 1000",
    "gap_over_3650_days": f"{DAYS} > 3650",
}) for t in TX}, axis=1), note="450 days is the longest plan duration observed in v2 (see 4.2); the thresholds are informational.")

target = con.sql(f"SELECT msno FROM tx_all ORDER BY {DAYS} DESC NULLS LAST LIMIT 1").fetchone()[0]
member = con.execute(f"""
    SELECT release, transaction_date, membership_expire_date, payment_plan_days, is_cancel,
           date_diff('day', lag(TRY_STRPTIME(membership_expire_date, '%Y%m%d')) OVER w,
                     TRY_STRPTIME(membership_expire_date, '%Y%m%d')) AS expiry_increment_days
    FROM tx_all WHERE msno = ?
    WINDOW w AS (ORDER BY transaction_date, membership_expire_date)
    ORDER BY transaction_date, membership_expire_date
""", [target]).df()
comparable = member["expiry_increment_days"].notna()
matches = member["expiry_increment_days"].eq(member["payment_plan_days"].astype(float))
show("Case study: the member with the largest gap — increment versus plan days", pd.Series({
    "records": len(member), "compared_rows": comparable.sum(),
    "matching_rows": (comparable & matches).sum(), "mismatching_rows": (comparable & ~matches).sum()}).to_frame("value"),
     note="Increment = this row's expiration date minus the previous row's, after sorting by transaction date and expiration date.")
show("Case study: records in order", member, max_rows=100)

**Ten largest `days_to_expiry` in tx_v1**

,member,payment_plan_days,actual_amount_paid,transaction_date,membership_expire_date,days_to_expiry,is_cancel
0,6oXmSsCk,410,1788,20150105,20170329,814,0
1,DRd9OxaJ,30,119,20150106,20170328,812,0
2,IwE/pih8,30,149,20150102,20170323,811,0
3,z1Lm/BlR,30,149,20150102,20170323,811,0
4,d81mjiF7,30,119,20150111,20170331,810,0
5,/Fpm8jjt,30,119,20150101,20170320,809,0
6,dDT8t5/g,30,119,20150105,20170324,809,0
7,OYy2aBqo,30,119,20150102,20170318,806,0
8,Jt/DLVnW,30,149,20150114,20170330,806,0
9,rb6Fy5tn,30,119,20150109,20170325,806,0


**Ten largest `days_to_expiry` in tx_v2**

,member,payment_plan_days,actual_amount_paid,transaction_date,membership_expire_date,days_to_expiry,is_cancel
0,F9//z42O,410,1788,20161017,20361015,"7,303",0
1,F9//z42O,410,1788,20161017,20350901,"6,893",0
2,F9//z42O,410,1788,20161017,20340718,"6,483",0
3,F9//z42O,410,1788,20161017,20330603,"6,073",0
4,F9//z42O,410,1788,20161017,20320419,"5,663",0
5,F9//z42O,410,1788,20161017,20310306,"5,253",0
6,F9//z42O,410,1788,20161017,20300120,"4,843",0
7,F9//z42O,30,149,20161017,20281206,"4,433",0
8,F9//z42O,30,149,20161017,20281106,"4,403",0
9,F9//z42O,30,149,20161017,20281007,"4,373",0


**Rows above fixed `days_to_expiry` thresholds**  
*450 days is the longest plan duration observed in v2 (see 4.2); the thresholds are informational.*

**Case study: the member with the largest gap — increment versus plan days**  
*Increment = this row's expiration date minus the previous row's, after sorting by transaction date and expiration date.*

,value
records,78
compared_rows,77
matching_rows,75
mismatching_rows,2


**Case study: records in order**

,release,transaction_date,membership_expire_date,payment_plan_days,is_cancel,expiry_increment_days
0,v1,20150215,20150817,180,0,
1,v1,20150831,20160227,180,0,194
2,v2,20160915,20171030,410,0,611
3,v2,20160915,20171129,30,0,30
4,v2,20160915,20171229,30,0,30
5,v2,20160915,20180128,30,0,30
6,v2,20160915,20180227,30,0,30
7,v2,20160915,20180329,30,0,30
8,v2,20160915,20180428,30,0,30
9,v2,20160915,20190612,410,0,410


### 4.6 Zero-day plans and payment context

**Purpose:** examine rows with `payment_plan_days = 0` (a zero duration does not mean zero payment) and where they occur.

**Logic:** the first table crosses cancellation with zero payment among zero-day rows; the second lists the most common (list price, amount paid) pairs; the third shows payment methods that have zero-day rows, with their share of that method's transactions.

**What to look for:** whether zero-day rows carry real payments; whether one payment method dominates only because of its volume (compare the percentage with the count).

In [51]:
zero_day = "TRY_CAST(payment_plan_days AS DOUBLE) = 0"

show("Zero-day rows: cancellation by zero payment", con.sql(f"""
    SELECT release, is_cancel, TRY_CAST(actual_amount_paid AS DOUBLE) = 0 AS is_zero_payment, count(*) AS rows
    FROM tx_all WHERE {zero_day} GROUP BY ALL ORDER BY ALL
""").df().set_index(["release", "is_cancel", "is_zero_payment"]))
show("Zero-day rows: most common list price and amount paid", con.sql(f"""
    SELECT release, plan_list_price, actual_amount_paid, count(*) AS rows
    FROM tx_all WHERE {zero_day} GROUP BY ALL ORDER BY rows DESC LIMIT 15
""").df().set_index(["release", "plan_list_price", "actual_amount_paid"]))
show("Payment methods with zero-day rows", con.sql(f"""
    SELECT release, payment_method_id, count(*) AS total_transactions,
           count(*) FILTER (WHERE {zero_day}) AS zero_day_transactions,
           100.0 * count(*) FILTER (WHERE {zero_day}) / count(*) AS zero_day_percentage
    FROM tx_all GROUP BY ALL HAVING zero_day_transactions > 0
    ORDER BY zero_day_transactions DESC LIMIT 15
""").df().set_index(["release", "payment_method_id"]))

**Zero-day rows: cancellation by zero payment**

**Zero-day rows: most common list price and amount paid**

**Payment methods with zero-day rows**

### 4.7 Relationship between `transactions` (v1) and `transactions_v2`

**Purpose:** decide whether the two releases can be stacked. If v2 were a superset, stacking would double count; the documented description says v1 ends 2017-02-28 and v2 is the March refresh, but 4.3 shows whether v2 carries earlier rows.

**Logic:** (1) identical rows across the two files, using `INTERSECT` on all nine fields. (2) Share of users in each release who also appear in the other (`coverage`). (3) For v2 rows dated before March 2017, how many share a (`msno`, `transaction_date`) key with a v1 row. (4) For every (`msno`, `transaction_date`) key present in both releases, whether the two rows agree on expiration date.

**What to look for:** zero identical rows with a nonzero number of shared keys means the releases overlap in time but disagree on the same user-day, so stacking them needs an explicit rule (for example, prefer one release for the overlap) before sequences are rebuilt. Expect high user coverage because v2 users are mostly existing members.

In [52]:
cols = ", ".join(["msno", "payment_method_id", "payment_plan_days", "plan_list_price", "actual_amount_paid",
                  "is_auto_renew", "transaction_date", "membership_expire_date", "is_cancel"])

show("Identical rows across the two releases", con.sql(f"""
    SELECT (SELECT count(*) FROM tx_v1) AS rows_v1, (SELECT count(*) FROM tx_v2) AS rows_v2,
           (SELECT count(*) FROM (SELECT {cols} FROM tx_v1 INTERSECT SELECT {cols} FROM tx_v2)) AS distinct_rows_in_both
""").df().T.rename(columns={0: "value"}))
show("User coverage: v2 users also in v1", ac.coverage(con, "tx_v2", "tx_v1", "msno"))
show("User coverage: v1 users also in v2", ac.coverage(con, "tx_v1", "tx_v2", "msno"))
show("v2 rows dated before March 2017 that share a (`msno`, `transaction_date`) key with v1", con.sql("""
    SELECT count(*) AS v2_rows_before_march,
           count(*) FILTER (WHERE EXISTS (SELECT 1 FROM tx_v1 a
                                          WHERE a.msno = b.msno AND a.transaction_date = b.transaction_date)) AS rows_with_key_in_v1
    FROM tx_v2 b WHERE b.transaction_date < '20170301'
""").df().T.rename(columns={0: "value"}))
show("Keys (`msno`, `transaction_date`) present in both releases: do the expiration dates agree?", con.sql("""
    SELECT count(DISTINCT (a.msno, a.transaction_date)) AS keys_in_both,
           count(*) AS row_pairs,
           count(*) FILTER (WHERE a.membership_expire_date = b.membership_expire_date) AS pairs_same_expiry,
           count(*) FILTER (WHERE a.membership_expire_date <> b.membership_expire_date) AS pairs_different_expiry
    FROM tx_v1 a JOIN tx_v2 b ON a.msno = b.msno AND a.transaction_date = b.transaction_date
""").df().T.rename(columns={0: "value"}))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Identical rows across the two releases**

,value
rows_v1,"21,547,746"
rows_v2,"1,431,009"
distinct_rows_in_both,0


**User coverage: v2 users also in v1**

,value
left_keys,"1,197,050"
matched,"1,134,533"
unmatched,"62,517"
matched_percentage,94.78%


...

,value
left_keys,"2,363,626"
matched,"1,134,533"
unmatched,"1,229,093"
matched_percentage,48.00%


**v2 rows dated before March 2017 that share a (`msno`, `transaction_date`) key with v1**

,value
v2_rows_before_march,"361,187"
rows_with_key_in_v1,"9,967"


**Keys (`msno`, `transaction_date`) present in both releases: do the expiration dates agree?**

,value
keys_in_both,"7,249"
row_pairs,"42,257"
pairs_same_expiry,0
pairs_different_expiry,"42,257"


#### 4.7.1 What the pre-March rows of `transactions_v2` add (exploratory)

**Purpose:** the monthly row counts show that v2 is not "v1 plus March": before March 2017 v2 holds far fewer rows than v1 (1,169 against 576,349 in January 2015), and 361,187 v2 rows are dated before March 2017. These checks ask who those rows belong to, where they sit relative to the same user's v1 history, and whether the rows of the 7,249 keys present in both releases fit together.

**Logic:** (1) coverage of the users who carry pre-March v2 rows in v1 and in the label and test tables; (2) for each pre-March v2 row, the position relative to the same user's latest earlier v1 transaction date (inside an active membership period, after an expiration gap, or with no earlier v1 row), plus their plan and cancellation profile; (3) for the shared keys, how many rows each release holds, whether a key and release share one plan, and whether adjacent rows (sorted by expiration date) differ by the plan duration.

**What to look for:** a large share of rows inside active v1 membership periods or after gaps, a plan mix very different from v1, and adjacent rows from different releases that still follow the plan-duration increment. Together these are consistent with v2 supplying records that v1 lacks; they do not prove it, and they do not show that v1 plus v2 is a complete history.

In [53]:
con.execute("CREATE OR REPLACE TEMP TABLE v2_early AS SELECT * FROM tx_v2 WHERE transaction_date < '20170301'")
con.execute("CREATE OR REPLACE TEMP TABLE v2_early_users AS SELECT DISTINCT msno FROM v2_early")
con.execute("CREATE OR REPLACE TEMP TABLE v2_march_users AS SELECT DISTINCT msno FROM tx_v2 WHERE transaction_date >= '20170301'")
con.execute("""CREATE OR REPLACE TEMP TABLE v2_early_only_users AS
               SELECT msno FROM v2_early_users WHERE msno NOT IN (SELECT msno FROM v2_march_users)""")

groups = {"users with pre-March v2 rows": "v2_early_users",
          "of which: no March row (early-only)": "v2_early_only_users",
          "users with March v2 rows": "v2_march_users"}
rows = []
for label, table in groups.items():
    for reference in ["tx_v1", "train_v1", "train_v2", "sub_v2"]:
        c = ac.coverage(con, table, reference, "msno")["value"]
        rows.append({"user group": label, "found in": reference, "users": c["left_keys"],
                     "found": c["matched"], "matched_percentage": c["matched_percentage"]})

show("Which users carry the pre-March v2 rows", pd.DataFrame(rows).set_index(["user group", "found in"]),
     note="Share of each user group that also appears in the reference table.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Which users carry the pre-March v2 rows**  
*Share of each user group that also appears in the reference table.*

In [54]:
con.execute("""CREATE OR REPLACE TEMP TABLE v1_for_early_users AS
               SELECT msno, transaction_date, max(membership_expire_date) AS membership_expire_date
               FROM tx_v1 WHERE msno IN (SELECT msno FROM v2_early_users)
               GROUP BY msno, transaction_date""")

position = con.sql("""
    SELECT CASE WHEN a.transaction_date IS NULL THEN '3 no earlier or same-day v1 row'
                WHEN a.membership_expire_date < b.transaction_date THEN '2 follows a v1 expiration gap'
                ELSE '1 falls inside an active v1 membership period' END AS position_relative_to_v1,
           count(*) AS rows,
           round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS percentage
    FROM v2_early b
    ASOF LEFT JOIN v1_for_early_users a ON a.msno = b.msno AND a.transaction_date <= b.transaction_date
    GROUP BY 1 ORDER BY 1
""").df().set_index("position_relative_to_v1")
show("Where each pre-March v2 row falls relative to the same user's previous v1 row", position,
     note="Previous v1 date = the user's latest v1 transaction date on or before the v2 row; the latest expiration date on that date is used.")

plans = (ac.value_counts_pct(con, "v2_early", "payment_plan_days").set_index("value")[["percentage"]]
         .add_suffix("_v2_pre_march")
         .join(ac.value_counts_pct(con, "tx_v1", "payment_plan_days").set_index("value")[["percentage"]]
               .add_suffix("_v1_all"), how="left")
         .head(6))
show("Plan durations: pre-March v2 rows versus all v1 rows (top 6 by v2 share)", plans)

cancel = pd.concat({t: ac.value_counts_pct(con, t, "is_cancel").set_index("value")["percentage"]
                    for t in ["v2_early", "tx_v1"]}, axis=1).add_prefix("percentage_")
show("Cancellation flag: pre-March v2 rows versus all v1 rows", cancel)

**Where each pre-March v2 row falls relative to the same user's previous v1 row**  
*Previous v1 date = the user's latest v1 transaction date on or before the v2 row; the latest expiration date on that date is used.*

,rows,percentage
position_relative_to_v1,,
1 falls inside an active v1 membership period,"227,254",62.92%
2 follows a v1 expiration gap,"92,696",25.66%
3 no earlier or same-day v1 row,"41,237",11.42%


**Plan durations: pre-March v2 rows versus all v1 rows (top 6 by v2 share)**

,percentage_v2_pre_march,percentage_v1_all
value,,
30,49.74%,87.97%
410,20.96%,0.37%
195,6.70%,0.51%
180,5.49%,0.24%
90,3.69%,0.06%
7,2.96%,2.68%


**Cancellation flag: pre-March v2 rows versus all v1 rows**

,percentage_v2_early,percentage_tx_v1
value,,
0,99.04%,96.02%
1,0.96%,3.98%


In [55]:
con.execute("""CREATE OR REPLACE TEMP TABLE shared_keys AS
    SELECT DISTINCT a.msno, a.transaction_date FROM tx_v1 a
    JOIN (SELECT DISTINCT msno, transaction_date FROM v2_early) b USING (msno, transaction_date)""")
con.execute("""CREATE OR REPLACE TEMP TABLE shared_rows AS
    SELECT 'v1' AS release, t.* FROM tx_v1 t JOIN shared_keys USING (msno, transaction_date)
    UNION ALL
    SELECT 'v2' AS release, t.* FROM tx_v2 t JOIN shared_keys USING (msno, transaction_date)""")

show("Rows on each side of the keys present in both releases", con.sql("""
    SELECT release, count(*) AS n_rows, count(DISTINCT (msno, transaction_date)) AS n_keys,
           round(count(*) * 1.0 / count(DISTINCT (msno, transaction_date)), 2) AS rows_per_key
    FROM shared_rows GROUP BY release ORDER BY release
""").df().set_index("release"))

show("Do the rows of one key and release share a single plan, amount, and cancel flag?", con.sql("""
    WITH g AS (SELECT msno, transaction_date, release,
                      count(DISTINCT payment_plan_days) AS plans, count(DISTINCT actual_amount_paid) AS amounts,
                      count(DISTINCT is_cancel) AS cancels
               FROM shared_rows GROUP BY ALL)
    SELECT count(*) AS key_release_groups,
           count(*) FILTER (WHERE plans = 1 AND amounts = 1 AND cancels = 1) AS single_plan_amount_cancel,
           round(100.0 * count(*) FILTER (WHERE plans = 1 AND amounts = 1 AND cancels = 1) / count(*), 2) AS percentage
    FROM g
""").df().T.rename(columns={0: "value"}))

show("Expiration increment versus plan days for adjacent rows within a key (sorted by expiration date)", con.sql("""
    WITH o AS (
        SELECT release, payment_plan_days::INT AS plan_days, TRY_STRPTIME(membership_expire_date, '%Y%m%d') AS expiry,
               lag(TRY_STRPTIME(membership_expire_date, '%Y%m%d')) OVER w AS previous_expiry,
               lag(release) OVER w AS previous_release
        FROM shared_rows
        WINDOW w AS (PARTITION BY msno, transaction_date ORDER BY membership_expire_date))
    SELECT CASE WHEN previous_release = release THEN '1 adjacent rows, same release'
                ELSE '2 adjacent rows, different releases' END AS pair_type,
           count(*) AS pairs,
           count(*) FILTER (WHERE date_diff('day', previous_expiry, expiry) = plan_days) AS increment_equals_plan_days,
           round(100.0 * count(*) FILTER (WHERE date_diff('day', previous_expiry, expiry) = plan_days) / count(*), 1) AS percentage
    FROM o WHERE previous_release IS NOT NULL GROUP BY 1 ORDER BY 1
""").df().set_index("pair_type"), note="Exploratory: a match means the later row extends the earlier row's expiration by its plan duration.")

**Rows on each side of the keys present in both releases**

,n_rows,n_keys,rows_per_key
release,,,
v1,"9,711","7,249",1.34
v2,"9,967","7,249",1.37


**Do the rows of one key and release share a single plan, amount, and cancel flag?**

,value
key_release_groups,"14,498"
single_plan_amount_cancel,"14,179"
percentage,97.80%


**Expiration increment versus plan days for adjacent rows within a key (sorted by expiration date)**  
*Exploratory: a match means the later row extends the earlier row's expiration by its plan duration.*

,pairs,increment_equals_plan_days,percentage
pair_type,,,
"1 adjacent rows, same release","5,180","4,779",92.30%
"2 adjacent rows, different releases","7,249","4,610",63.60%


In [56]:
fields = ["membership_expire_date", "actual_amount_paid", "plan_list_price", "payment_method_id",
          "payment_plan_days", "is_auto_renew", "is_cancel"]
differ = ", ".join(f'count(*) FILTER (WHERE a."{f}" <> b."{f}") AS "{f}"' for f in fields)
others_equal = " AND ".join(f'a."{f}" = b."{f}"' for f in fields if f != "membership_expire_date")

counts = con.sql(f"""
    SELECT count(*) AS row_pairs, {differ},
           count(*) FILTER (WHERE {others_equal}) AS only_expiry_differs
    FROM (SELECT * FROM shared_rows WHERE release = 'v1') a
    JOIN (SELECT * FROM shared_rows WHERE release = 'v2') b USING (msno, transaction_date)
""").df().iloc[0]

row_pairs = int(counts["row_pairs"])
field_differences = pd.DataFrame({"pairs": counts[fields + ["only_expiry_differs"]].astype(int)})
field_differences["percentage"] = field_differences["pairs"] / row_pairs * 100
show(f"Fields that differ across the {row_pairs:,} v1-v2 row pairs of the shared keys", field_differences,
     note="A pair = one v1 row and one v2 row with the same user and transaction date. `only_expiry_differs` counts pairs where every other field is equal.")


**Fields that differ across the 42,257 v1-v2 row pairs of the shared keys**  
*A pair = one v1 row and one v2 row with the same user and transaction date. `only_expiry_differs` counts pairs where every other field is equal.*

,pairs,percentage
membership_expire_date,"42,257",100.00%
actual_amount_paid,"3,853",9.12%
plan_list_price,"3,786",8.96%
payment_method_id,"3,118",7.38%
payment_plan_days,"2,933",6.94%
is_auto_renew,"1,693",4.01%
is_cancel,"1,672",3.96%
only_expiry_differs,"37,856",89.59%


### 4.8 Findings, impact, and decisions

Full-data counts reproduced by a read-only run of the cells above (release-relationship details come from 4.7.1); re-run the notebook to confirm them in your kernel. *Hypothesis* means not verified.

| Area | Finding | Impact and decision |
|---|---|---|
| Structure | v1: 21,547,746 rows, 2,363,626 users. v2: 1,431,009 rows, 1,197,050 users. Nine fields, none missing | Many records per user: aggregate to the intended grain before any user-level join |
| Duplicates and grain | Exact duplicates: v1 3,339 excess rows, v2 0. Same user and day: 286,404 (v1) and 33,292 (v2) excess rows, up to 48 and 131 per key | (`msno`, `transaction_date`) is not a unique key. Do not drop duplicates or de-duplicate on user and day; investigate before the Silver layer (open) |
| Values | Flags only 0 and 1; no non-numeric or negative values; plan days 0–450; prices 0–2,000. Zero-day plans: v1 870,124 (4.04%), v2 2,218 (0.15%), almost all with a nonzero payment | Zero duration is not zero payment: do not fill durations or erase payments. Cancellation is not a churn label |
| Dates | No invalid dates. v1: 2015-01-01 to 2017-02-28. v2: 2015-01-01 to 2017-03-31 (1,069,822 rows in March, 361,187 before). Expiration before 2015: 9,840 v1 rows (1,776 on 1970-01-01), none in v2 | v2 is not a March-only extract. Epoch-like dates would distort duration arithmetic: preserve, flag in Silver, confirm the cause (open) |
| Expiry before transaction | v1: 153,660 rows (147,200 cancellations = 17.18% of cancellations; 6,460 non-cancellations). v2: 5,106 (5,104 cancellations, 2 non-cancellations) | Mostly cancellation records, so blanket deletion would remove them. Preserve and flag; cause unconfirmed |
| Distant expiry | Largest gap 814 days (v1) and 7,303 days (v2, one member); 19,357 v2 rows above 1,000 days, none in v1. That member's 78 records: 75 of 77 expiry increments equal plan days | Consistent with stacked renewals, but one member is not population evidence. Do not cap distant dates on this alone |
| Release relationship | 0 identical rows. 94.78% of v2 users are in v1 (48.00% of v1 users in v2). The 361,187 pre-March v2 rows lie 62.9% inside an active v1 membership, 25.7% after a v1 expiry gap, 11.4% with no earlier v1 row, and skew toward long plans and non-cancellations. 7,249 (`msno`, `transaction_date`) keys occur in both releases (9,711 v1 rows, 9,967 v2 rows); 89.59% of their 42,257 row pairs differ only in expiration date, and adjacent rows follow the plan-duration increment in 63.6% of cross-release pairs (92.3% within one release) | The pre-March v2 rows look like a supplement to v1 and the shared keys look complementary (same-day stacked purchases) rather than conflicting (*hypothesis*). Stack as `tx_all` with a `release` column and never drop rows; handle the unexplained 36% when rebuilding membership sequences |

**Open:** 
- cause of the v1 exact duplicates and of the 1970-01-01 dates; 
- why v2 carries the pre-March rows; whether v1 plus v2 is a complete history; 
- plan and cancellation semantics; training-user transaction coverage; observation cutoffs. 

No automatic deletion, imputation, or de-duplication has been applied.


## 5. User logs — `user_logs.csv` (v1) and `user_logs_v2.csv` (v2)

**Purpose:** Daily listening summaries. 

**Grain:** one row per `msno` and `date` (verified in 5.1 and 5.5).

**Source fields:** `msno`, `date`, `num_25`, `num_50`, `num_75`, `num_985`, `num_100`, `num_unq`, `total_secs`.

**Method:** both releases are checked separately and side by side (`logs_v1`, `logs_v2`), then together through `logs_all` (a `release` column plus the nine fields) where one query must cover both. They are read from text-typed Parquet copies of the CSVs (row counts verified against the CSVs at conversion), so each check interprets numbers and dates itself and source values are never altered. The v1 table has about 392 million rows, so some cells take minutes.


### 5.1 Structure, key uniqueness, and completeness

**Purpose:** confirm the shape of each release, that (`msno`, `date`) identifies one row, and that no value is missing.

**Logic:** `table_summary` counts rows, columns, and distinct users; `key_uniqueness` checks the composite key within each release; `missingness` counts NULL or blank values per column.

**What to look for:** `excess_rows = 0` and `missing_key_rows = 0` (a unique, complete key) and no missing value in any column.


In [57]:
LOGS = ["logs_v1", "logs_v2"]

show("Structure: rows, columns, distinct users",
     pd.concat({t: ac.table_summary(con, t, key="msno")["value"] for t in LOGS}, axis=1))
show("Key uniqueness: (`msno`, `date`) must identify one row",
     pd.concat({t: ac.key_uniqueness(con, t, ["msno", "date"])["value"] for t in LOGS}, axis=1),
     note="excess_rows counts rows beyond the first for a repeated key; 0 means the key is unique within the release.")
show("Missing values per column (NULL or blank)",
     pd.concat({t: ac.missingness(con, t)["missing"] for t in LOGS}, axis=1))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Structure: rows, columns, distinct users**

,logs_v1,logs_v2
rows,"392,106,543","18,396,362"
columns,9,9
unique_msno,"5,234,111","1,103,894"


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Key uniqueness: (`msno`, `date`) must identify one row**  
*excess_rows counts rows beyond the first for a repeated key; 0 means the key is unique within the release.*

,logs_v1,logs_v2
rows,"392,106,543","18,396,362"
missing_key_rows,0,0
unique_keys,"392,106,543","18,396,362"
duplicated_keys,0,0
excess_rows,0,0
rows_in_duplicated_keys,0,0
max_rows_per_key,1,1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Missing values per column (NULL or blank)**

,logs_v1,logs_v2
msno,0,0
date,0,0
num_25,0,0
num_50,0,0
num_75,0,0
num_985,0,0
num_100,0,0
num_unq,0,0
total_secs,0,0


### 5.2 Date validity and monthly coverage

**Purpose:** verify that `date` holds real `YYYYMMDD` dates and see which months each release covers.

**Logic:** `date_validity` reports missing and invalid dates and the range; the monthly table counts log rows and distinct active users per calendar month and release.

**What to look for:** no invalid dates; v1 ending on 2017-02-28 and v2 covering only March 2017; no missing months.


In [58]:
show("Date validity and range", pd.concat({t: ac.date_validity(con, t, ["date"]) for t in LOGS}))

monthly = con.sql("""
    SELECT release, substr(date, 1, 4) || '-' || substr(date, 5, 2) AS month,
           count(*) AS rows, count(DISTINCT msno) AS users
    FROM logs_all GROUP BY ALL ORDER BY month
""").df().set_index("month")
show("Log rows and active users per month and release", monthly, max_rows=40,
     note="Users are distinct within each month; a user active in several months is counted in each.")


**Date validity and range**

,,missing,invalid,earliest,latest
logs_v1,date,0,0,2015-01-01,2017-02-28
logs_v2,date,0,0,2017-03-01,2017-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Log rows and active users per month and release**  
*Users are distinct within each month; a user active in several months is counted in each.*

,release,rows,users
month,,,
2015-01,v1,"12,897,706","937,789"
2015-02,v1,"11,457,186","933,040"
2015-03,v1,"13,233,548","944,739"
2015-04,v1,"13,146,949","939,930"
2015-05,v1,"13,477,134","924,216"
2015-06,v1,"13,111,785","916,862"
2015-07,v1,"13,186,484","871,491"
2015-08,v1,"13,428,113","920,129"
2015-09,v1,"13,559,390","903,194"


### 5.3 Numeric validity and identifier format

**Purpose:** check that the count and duration fields parse and have sensible ranges, and that identifiers are clean.

**Logic:** `numeric_range` reports missing, non-numeric, negative, and zero counts and the minimum and maximum per column; `flag_counts` tests identifier padding and length. Zero is legitimate for the play-band counts; `num_unq` should be at least 1.

**What to look for:** `non_numeric = 0`; negative values only in `total_secs`; very large maxima that need review (listening seconds are examined in 5.4); no padded or odd-length `msno`.


In [59]:
NUM = ["num_25", "num_50", "num_75", "num_985", "num_100", "num_unq", "total_secs"]

for t in LOGS:
    show(f"Numeric range in {t}", ac.numeric_range(con, t, NUM).T)
show("`msno` format flags", pd.concat({t: ac.flag_counts(con, t, {
    "msno_blank_or_padded": "msno = '' OR msno <> trim(msno)",
    "msno_length_not_44": "length(msno) <> 44",
}) for t in LOGS}, axis=1), note="Informational: identifiers are expected to be 44-character strings without padding.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Numeric range in logs_v1**

,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs
missing,0,0,0,0,0,0,0
non_numeric,0,0,0,0,0,0,0
negative,0,0,0,0,0,0,"61,493"
zero,"102,161,124","187,960,924","212,940,691","208,383,075","14,430,534",0,0
min,0,0,0,0,0,1,"-9,223,372,036,854,776"
max,"18,798","1,710","1,690","2,747","42,004","4,784","9,223,372,036,854,776"


**Numeric range in logs_v2**

,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs
missing,0,0,0,0,0,0,0
non_numeric,0,0,0,0,0,0,0
negative,0,0,0,0,0,0,0
zero,"4,905,984","9,134,553","10,273,800","10,013,950","504,974",0,0
min,0,0,0,0,0,1,0.001
max,"5,639",912,508,"1,561","41,107","4,925","9,194,058.52"


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**`msno` format flags**  
*Informational: identifiers are expected to be 44-character strings without padding.*

### 5.4 `total_secs` plausibility

**Purpose:** size the negative and implausibly large listening durations before any aggregation, because one extreme value would dominate sums and averages.

**Logic:** rows are placed in bands by `total_secs`: *negative, within one day (86,400 seconds), one to seven days, seven days to 10^12 seconds*, and *above 10^12*. The one-day limit is a review threshold, not a cleaning rule. The second table counts rows outside the one-day range per month.

**What to look for:** the share of rows outside the plausible band, how many distinct users they involve, and whether they cluster in particular months.


In [60]:
SECS = 'TRY_CAST(total_secs AS DOUBLE)'

bands = pd.concat({t: con.sql(f"""
    SELECT CASE WHEN {SECS} < 0        THEN '1 negative'
                WHEN {SECS} <= 86400   THEN '2 within one day'
                WHEN {SECS} <= 604800  THEN '3 one to seven days'
                WHEN {SECS} <= 1e12    THEN '4 seven days to 1e12'
                ELSE                        '5 above 1e12' END AS band,
           count(*) AS rows, count(DISTINCT msno) AS users, min({SECS}) AS min_secs, max({SECS}) AS max_secs
    FROM {t} GROUP BY band ORDER BY band
""").df().set_index("band") for t in LOGS})
show("`total_secs` by plausibility band", bands,
     note="86,400 seconds is one day: a review threshold, not a cleaning rule. Source values are kept.")

bad = con.sql(f"""
    SELECT release, substr(date, 1, 4) || '-' || substr(date, 5, 2) AS month, count(*) AS bad_rows
    FROM logs_all WHERE {SECS} < 0 OR {SECS} > 86400 GROUP BY ALL ORDER BY month
""").df().set_index("month")
show("Rows with negative or above-one-day `total_secs`, per month", bad, max_rows=40)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**`total_secs` by plausibility band**  
*86,400 seconds is one day: a review threshold, not a cleaning rule. Source values are kept.*

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Rows with negative or above-one-day `total_secs`, per month**

,release,bad_rows
month,,
2015-01,v1,"7,229"
2015-02,v1,"7,368"
2015-03,v1,"9,144"
2015-04,v1,"6,911"
2015-05,v1,"12,309"
2015-06,v1,"18,722"
2015-07,v1,"26,640"
2015-08,v1,"22,241"
2015-09,v1,"18,087"


### 5.5 Cross-release checks: dates and users

**Purpose:** confirm the two releases can be stacked without repeated keys, and measure how many users appear in both.

**Logic:** a (`msno`, `date`) key can repeat across releases only if a date occurs in both, so the first table counts dates present in both. (A direct grouping of all rows of both releases exceeded the memory limit and is not needed.) `coverage` then measures user overlap in both directions.

**What to look for:** `dates_in_both = 0`, and what share of March users were already active before March.


In [61]:
show("Dates present in both releases", con.sql("""
    SELECT (SELECT count(DISTINCT date) FROM logs_v1) AS distinct_dates_v1,
           (SELECT count(DISTINCT date) FROM logs_v2) AS distinct_dates_v2,
           (SELECT count(*) FROM (SELECT DISTINCT date FROM logs_v1) a
            JOIN (SELECT DISTINCT date FROM logs_v2) b USING (date)) AS dates_in_both
""").df().T.rename(columns={0: "value"}),
     note="Keys are (`msno`, `date`). With no date in both releases, a key cannot repeat across them, so uniqueness within each release (5.1) implies uniqueness across the two.")

show("User overlap between the two releases", pd.concat({
    "v2 users also in v1": ac.coverage(con, "logs_v2", "logs_v1", "msno")["value"],
    "v1 users also in v2": ac.coverage(con, "logs_v1", "logs_v2", "msno")["value"]}, axis=1))


**Dates present in both releases**  
*Keys are (`msno`, `date`). With no date in both releases, a key cannot repeat across them, so uniqueness within each release (5.1) implies uniqueness across the two.*

,value
distinct_dates_v1,790
distinct_dates_v2,31
dates_in_both,0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**User overlap between the two releases**

,v2 users also in v1,v1 users also in v2
left_keys,"1,103,894","5,234,111"
matched,"998,583","998,583"
unmatched,"105,311","4,235,528"
matched_percentage,90.46%,19.08%


### 5.6 Coverage of label tables and test rosters by logs

**Purpose:** measure how many labeled and test users have any listening log row.

**Logic:** `coverage` left-joins the distinct users of each table to the distinct log users of each release and of both together (`logs_all`). "Found" means at least one log row in that release; no observation window has been applied.

**What to look for:** the share of users without any log row and whether it is similar across tables. Users without logs have no behavioral features; absence of logs must not be read as zero listening.


In [62]:
rows = []
for table in ["train_v1", "train_v2", "sub_zero", "sub_v2"]:
    for reference in ["logs_v1", "logs_v2", "logs_all"]:
        c = ac.coverage(con, table, reference, "msno")["value"]
        rows.append({"table": table, "logs in": reference, "users": c["left_keys"],
                     "found": c["matched"], "matched_percentage": c["matched_percentage"]})
show("Coverage: share of each table's users with at least one log row",
     pd.DataFrame(rows).set_index(["table", "logs in"]),
     note="'At least one log row' in any month of the release; no observation window has been applied.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Coverage: share of each table's users with at least one log row**  
*'At least one log row' in any month of the release; no observation window has been applied.*

### 5.7 Findings, impact, and decisions

Full-data counts reproduced by a read-only run of the cells above; re-run the notebook to confirm them in your kernel.

| Area | Finding | Impact and decision |
|---|---|---|
| Structure and grain | v1: 392,106,543 rows, 5,234,111 users; v2: 18,396,362 rows, 1,103,894 users; nine fields, none missing. (`msno`, `date`) is unique in each release and no date occurs in both | One row per user and day; the releases can be stacked as `logs_all` with a `release` column without repeated keys |
| Coverage in time | v1: 2015-01-01 to 2017-02-28; v2: 2017-03-01 to 2017-03-31; 27 consecutive months with 11.5–18.4 million rows and 0.87–1.18 million active users per month; no invalid dates | v2 is the March continuation of v1, not a replacement. Feature windows are fixed once cutoffs are defined |
| Counts | No missing, non-numeric, or negative values in the play-band counts; `num_unq` is at least 1. Extreme maxima: `num_100` 42,004 (v1) and 41,107 (v2); `num_unq` 4,784 and 4,925 | Plausibility is unconfirmed (possible automated play); preserve and review, do not cap |
| Identifiers | Every `msno` has 44 characters without padding | Safe to join on `msno` |
| `total_secs` anomalies | v1: 61,493 negative rows (29,929 users), 137,007 rows between one and seven days, 5,921 between seven days and 10^12 seconds, 65 above 10^12 (about 0.052% of rows). v2: 4,200 rows above one day (0.023%). Present in every month (v1 peak 26,640 in July 2015, about 4,000 per month from 2016) | One value near 9.2×10^15 would dominate any sum or mean. Preserve source values; in Silver, flag the rows and null the invalid durations while keeping the original (thresholds pending, especially the one-to-seven-day band) |
| User overlap | 998,583 users appear in both releases: 90.46% of v2 users and 19.08% of v1 users | About one in ten March users was not active before March |
| Coverage of label tables | At least one log row in either release: `train_v1` 87.76%, `train_v2` and `sub_zero` 88.07%, `sub_v2` 86.87%. In v2 alone: 74.51%, 77.71%, 77.71%, 77.16% | About 12–13% of labeled and test users have no log history: handle missing behavior explicitly; absence is not zero listening |

**Open:** 
- meaning of zero play-band counts and of the extreme maxima; 
- thresholds and treatment for `total_secs`; 
- whether users without logs are the same users without member profiles; 
- observation windows and cutoffs for features.
